Importamos librerías, cargamos los datos y chequeamos columnas y tipos de dato.

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [19]:
pd.set_option('display.max_columns',None)

In [20]:
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/df_tenis_columns.csv'

In [21]:
df = pd.read_csv(url)

In [22]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

In [23]:
df.columns

Index(['Unnamed: 0', 'ATP', 'Location', 'Tournament', 'Date', 'Series',
       'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank',
       'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4',
       'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'MaxW',
       'MaxL', 'AvgW', 'AvgL', 'Fecha', 'playerA', 'playerB', 'rankA', 'rankB',
       'PtsA', 'PtsB', 'B365A', 'B365B', 'MaxA', 'MaxB', 'AvgA', 'AvgB', 'A1',
       'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5', 'B5', 'setsA', 'setsB',
       'target'],
      dtype='object')

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16061 entries, 0 to 16060
Data columns (total 61 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  16061 non-null  int64         
 1   ATP         16061 non-null  float64       
 2   Location    16061 non-null  object        
 3   Tournament  16061 non-null  object        
 4   Date        16061 non-null  object        
 5   Series      16061 non-null  object        
 6   Court       16061 non-null  object        
 7   Surface     16061 non-null  object        
 8   Round       16061 non-null  object        
 9   Best of     16061 non-null  float64       
 10  Winner      16061 non-null  object        
 11  Loser       16061 non-null  object        
 12  WRank       16061 non-null  float64       
 13  LRank       16061 non-null  float64       
 14  WPts        16061 non-null  float64       
 15  LPts        16061 non-null  float64       
 16  W1          16061 non-

## Creación de Variables

Vamos a empezar a formar un montón de las variables que van a ser importantes para que después nuestro modelo de clasificación pueda decidir.

Hacemos una columna con cantidad total de sets de ese partido

In [25]:
df['setsPartido'] = df['setsA'] + df['setsB']

Implied Probabilities de cada campo que tiene cuotas (odds)

In [26]:
def implied_prob_adjusted(odds_A, odds_B):
    """Pasamos odds decimales en probabilidades ajustadas eliminando el margen del bookmaker."""
    inv_A = 1 / odds_A
    inv_B = 1 / odds_B

    total = inv_A + inv_B

    prob_A = inv_A / total
    prob_B = inv_B / total

    return prob_A, prob_B

In [27]:
probA = 1 / df['B365A']
probB = 1 / df['B365B']
df['B365BookmakersMargin'] = (probA + probB - 1)

df['B365ProbA'], df['B365ProbB'] = implied_prob_adjusted(df['B365A'], df['B365B'])
df['ProbAvgA'], df['ProbAvgB'] = implied_prob_adjusted(df['AvgA'], df['AvgB'])
df['ProbMaxA'], df['ProbMaxB'] = implied_prob_adjusted(df['MaxA'], df['MaxB'])

df['market_uncertainty'] = np.abs(df['ProbAvgA'] - 0.5) # Mas bajo, mas parejo el partido

# No hacemos diferencia entre probabilidades de victoria B365, Avg y Max, porque por su origen no son comparables
# Sí hacemos diferencia entre las cuotas de A y B, porque son comparables entre sí
df['OddsDiffB365'] = df['B365A'] - df['B365B']
df['OddsDiffAvg'] = df['AvgA'] - df['AvgB']
df['OddsDiffMax'] = df['MaxA'] - df['MaxB']

# Diferencia promedio de probabilidades
df['AvgOddsDiff'] = (df['ProbAvgA'] - df['ProbAvgB'])

# Probabilidades con un logaritmo aplicado, recomendado por ChatGPT
df['logit_oddsA'] = np.log(df['ProbAvgA'] / (1 - df['ProbAvgA']))
df['logit_oddsB'] = np.log(df['ProbAvgB'] / (1 - df['ProbAvgB']))

Ranking Difference, ATP Points Difference

In [28]:
df['rankDiff'] = df['rankA'] - df['rankB']
df['ptsDiff'] = df['PtsA'] - df['PtsB']

Porcentaje de partidos ganados en los últimos 5, 10 o 20 partidos

In [29]:
jug = pd.concat([df['playerA'], df['playerB']], axis=0).value_counts()
print(f'Total: {jug.shape[0]} jugadores')
print(f'Con mas de 20 partidos: {jug[jug>20].shape[0]} jugadores')
print(f'Con mas de 10 partidos: {jug[jug>10].shape[0]} jugadores')
print(f'Con mas de 5 partidos: {jug[jug>5].shape[0]} jugadores')

Total: 613 jugadores
Con mas de 20 partidos: 264 jugadores
Con mas de 10 partidos: 325 jugadores
Con mas de 5 partidos: 385 jugadores


In [30]:
# Hacemos un df con Jugadores A y otro con Jugadores B y los concatenamos uno abajo del otro
a = df[['Fecha','playerA','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerA':'Player','target':'win'})
b = df[['Fecha','playerB','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerB':'Player','target':'win'})
b['win'] = 1 - b['win']

matches_long = pd.concat([a, b], ignore_index=True).sort_values("Fecha").reset_index(drop=True)

In [31]:
# Calculamos winrate de los ultimos 5 partidos
matches_long['winrate_5'] = (matches_long.groupby('Player')['win'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

# Lo mismo pero de 10 partidos
matches_long['winrate_10'] = (matches_long.groupby('Player')['win'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()))

# Calculamos historicos
matches_long["matches_played_before"] = matches_long.groupby("Player").cumcount() # cumcount mira hasta la fila anterior
matches_long["wins_before"] = (matches_long.groupby("Player")["win"].transform(lambda x: x.cumsum().shift(1)).fillna(0)) # cumsum mira hasta la fila actual -> usamos shift
matches_long["win_pct_before"] = (matches_long["wins_before"] / matches_long["matches_played_before"].replace(0, pd.NA))

Sets jugados en ultimos 5 dias y sets jugados en ultimos 30 dias

In [32]:
def sets_ultimos_5_dias(group):
    resultados = []
    # Para cada partido de ese grupo (que tiene los partidos de un jugador)
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        # Calcula la fecha de ese partido - 5 dias
        limite = fecha - pd.Timedelta(days=5)
        # Mira la cantidad de partidos que hubo antes de la fecha del partido pero despues del limite (que es la fecha del partido - 5)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        # Agarra los partidos que fueron True y suma los sets de esos partidos
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

# Lo mismo que arriba pero cambiando la cantidad de dias
def sets_ultimos_30_dias(group):
    resultados = []
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        limite = fecha - pd.Timedelta(days=30)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

matches_long['sets_5d'] = (matches_long.groupby('Player').apply(sets_ultimos_5_dias, include_groups=False).reset_index(level=0, drop=True))
matches_long['sets_30d'] = (matches_long.groupby('Player').apply(sets_ultimos_30_dias, include_groups=False).reset_index(level=0, drop=True))

# Detalle: include_groups = False hace que no se pase la columna Player a la funcion, pero eso no es problema porque esa columna no la usamos

Partidos en ultimos 365 dias

In [33]:
def partidos_ultimos_365_dias(group):
  resultados = []
  # Para cada fecha de los partidos de un jugador:
  for current_match_date in group['Fecha']:
    # Agarramos la fecha del partido - 365 dias
    limite = current_match_date - pd.Timedelta(days=365)
    # Buscamos obtener con True los partidos que cumplen la condicion de haber sido 365 dias antes del partido
    mask = (group['Fecha'] < current_match_date) & (group['Fecha'] >= limite)
    # Contamos la cantidad de filas que cumplen esa condicion
    resultados.append(group.loc[mask].shape[0])
  return pd.Series(resultados, index=group.index)

matches_long['partidos_365d'] = matches_long.groupby('Player').apply(partidos_ultimos_365_dias, include_groups=False).reset_index(level=0, drop=True)

Dias desde el ultimo partido: se asume df ordenado por fecha

In [34]:
def dias_desde_ultimo_partido(group):
    resultados = []
    fechas = group['Fecha'].values
    # Para cada partido del groupby de jugador que entro a la funcion
    for i, current_match_date in enumerate(fechas):
        # Partidos ANTERIORES al partido en cuestion, en terminos de posición en el df, no solo en fecha
        anteriores = group['Fecha'].iloc[:i]

        if anteriores.empty:
            resultados.append(pd.NA)  # Si no hubo partidos antes, ponemos NaN
        else:
            resultados.append((current_match_date - anteriores.max()).days) # Si hubo, ponemos la diferencia entre la fecha el partido en cuestion y la fecha mas reciente a ese partido
    return pd.Series(resultados, index=group.index)

matches_long['dias_ultimo_partido'] = matches_long.groupby('Player').apply(dias_desde_ultimo_partido, include_groups=False).reset_index(level=0, drop=True)

Racha actual de victorias: se asume df ordenado por fecha

In [35]:
def racha_victorias(group):
    resultados = []
    # Para cada fila de un groupby de un jugador
    for i, current_match_date in enumerate(group['Fecha']):
      # Agarramos los partidos jugados antes de ese dia
      partidos_anteriores = group.iloc[:i].sort_values('Fecha', ascending=False)
      racha = 0
      for gano in partidos_anteriores['win']:
          if gano == 1:
              racha += 1
          else:
              break  # Se cortó la racha
      resultados.append(racha)
    return pd.Series(resultados, index=group.index)

matches_long['racha_victorias'] = matches_long.groupby('Player').apply(racha_victorias, include_groups=False).reset_index(level=0, drop=True)

In [36]:
# Podemos hacer una pequeña comprobación de que todo está OK
# matches_long[matches_long['Player'] == 'Nadal R.'].head(20)

Unimos todas las columnas anteriores creadas en matches_long al df original

In [37]:
FEATURE_COLS = ['winrate_5', 'winrate_10', "wins_before", "win_pct_before", 'sets_5d', 'sets_30d', 'partidos_365d', 'dias_ultimo_partido', 'racha_victorias']

# Hacemos un inner join entre el df de matches_long y la columna de jugadores A del df en base al id del partido y el nombre del jugador A
feats_A = (matches_long.merge(df[['Unnamed: 0', 'playerA']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerA'])
    [['Unnamed: 0'] + FEATURE_COLS].rename(columns={c: f'{c}_A' for c in FEATURE_COLS})) # (aca renombramos las columnas agregandole que son del jugador A)

# Hacemos un inner join entre el df de matches_long y la columna de jugadores B del df en base al id del partido y el nombre del jugador B
feats_B = (matches_long.merge(df[['Unnamed: 0', 'playerB']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerB'])
    [['Unnamed: 0'] + FEATURE_COLS].rename(columns={c: f'{c}_B' for c in FEATURE_COLS}))

# Mergeamos el df original con un left join con cada df de estadisticas: primero con el de A, despues con el de B
# Como feats_A y feats_B tienen la misma cantidad de filas que el df original (una por cada jugador A o jugador B) -> todo se une bien
df = df.merge(feats_A, on='Unnamed: 0', how='left')
df = df.merge(feats_B, on='Unnamed: 0', how='left')

Hacemos la diferencia entre las variables que acabamos de incorporar

In [38]:
df['diff_winrate_5'] = (df['winrate_5_A'] - df['winrate_5_B'])
df['diff_winrate_10'] = (df['winrate_10_A'] - df['winrate_10_B'])
df['diff_win_pct_before'] = (df['win_pct_before_A'] - df['win_pct_before_B'])
df['diff_sets_5d'] = (df['sets_5d_A'] - df['sets_5d_B'])
df['diff_sets_30d'] = (df['sets_30d_A'] - df['sets_30d_B'])
df['diff_dias_ultimo_partido'] = (df['dias_ultimo_partido_A'] - df['dias_ultimo_partido_B'])
df['diff_partidos_365d'] = (df['partidos_365d_A'] - df['partidos_365d_B'])

Cantidad de partidos previos entre ambos

In [39]:
# Cantidad de partidos previos entre ambos jugadores (Head-to-Head)
# Create a unique identifier for each match pair, sorted alphabetically to handle both playerA vs playerB and playerB vs playerA
df['h2h_pair_id'] = df.apply(lambda row: tuple(sorted([row['playerA'], row['playerB']])), axis=1)

print(f'Esta todo ok? {(df.groupby('h2h_pair_id')['Fecha'].apply(lambda x: x.duplicated().sum()).sum() == 0)}')

# Sort the DataFrame by this pair ID and then by date to ensure correct cumulative counting
df_sorted_h2h = df.sort_values(by=['h2h_pair_id', 'Fecha']).copy()

# Calculate the cumulative count of matches for each unique pair.
# cumcount() starts from 0, so it directly gives the number of previous matches for that pair.
df_sorted_h2h['h2h_matches_previous'] = df_sorted_h2h.groupby('h2h_pair_id').cumcount()

# Merge this new feature back into the original DataFrame using 'Unnamed: 0' (unique match ID)
df = df.merge(df_sorted_h2h[['Unnamed: 0', 'h2h_matches_previous']], on='Unnamed: 0', how='left')

# Drop the temporary column
df.drop(columns=['h2h_pair_id'], inplace=True)

Esta todo ok? True


Otras variables posibles:
- Record en cada superficie, previo al partido

Guardamos df sacando las columnas de ganador y perdedor o aquellas que no tiene más sentido que sigamos llevando porque no van a entrar al modelo.

In [40]:
columns_drop = ['Unnamed: 0', 'ATP', 'Tournament', 'Date', 'Winner', 'Loser', 'WRank',
       'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W',
       'B365L', 'MaxW', 'MaxL', 'AvgW', 'AvgL']

In [41]:
df = df.drop(columns=columns_drop)

In [42]:
df.to_csv('df_feat_eng.csv', index=False)

In [44]:
df

,Location,Series,Court,Surface,Round,Best of,Fecha,playerA,playerB,rankA,rankB,PtsA,PtsB,B365A,B365B,MaxA,MaxB,AvgA,AvgB,A1,B1,A2,B2,A3,B3,A4,B4,A5,B5,setsA,setsB,target,setsPartido,B365BookmakersMargin,B365ProbA,B365ProbB,ProbAvgA,ProbAvgB,ProbMaxA,ProbMaxB,market_uncertainty,OddsDiffB365,OddsDiffAvg,OddsDiffMax,AvgOddsDiff,logit_oddsA,logit_oddsB,rankDiff,ptsDiff,winrate_5_A,winrate_10_A,wins_before_A,win_pct_before_A,sets_5d_A,sets_30d_A,partidos_365d_A,dias_ultimo_partido_A,racha_victorias_A,winrate_5_B,winrate_10_B,wins_before_B,win_pct_before_B,sets_5d_B,sets_30d_B,partidos_365d_B,dias_ultimo_partido_B,racha_victorias_B,diff_winrate_5,diff_winrate_10,diff_win_pct_before,diff_sets_5d,diff_sets_30d,diff_dias_ultimo_partido,diff_partidos_365d,h2h_matches_previous
0,Doha,ATP250,Outdoor,Hard,1st Round,3.0,2015-01-05,Andujar P.,Gasquet R.,41.0,26.0,950.0,1350.0,4.33,1.20,4.84,1.26,4.15,1.21,3.0,6.0,5.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0,2.0,0.064280,0.216998,0.783002,0.225746,0.774254,0.206557,0.793443,0.274254,3.13,2.94,3.58,-0.548507,-1.232488,1.232488,15.0,-400.0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,<NA>,0.0,0.0,<NA>,0,0
1,Doha,ATP250,Outdoor,Hard,1st Round,3.0,2015-01-05,Dodig I.,Safwat M.,89.0,289.0,575.0,168.0,1.16,5.00,1.19,5.84,1.16,5.05,6.0,3.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1,2.0,0.062069,0.811688,0.188312,0.813205,0.186795,0.830725,0.169275,0.313205,-3.84,-3.89,-4.65,0.626409,1.470968,-1.470968,-200.0,407.0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,<NA>,0.0,0.0,<NA>,0,0
2,Doha,ATP250,Outdoor,Hard,1st Round,3.0,2015-01-05,Verdasco F.,Gabashvili T.,33.0,66.0,1135.0,730.0,1.40,2.75,1.47,3.09,1.41,2.81,4.0,6.0,6.0,3.0,7.0,6.0,0.0,0.0,0.0,0.0,2.0,1.0,1,3.0,0.077922,0.662651,0.337349,0.665877,0.334123,0.677632,0.322368,0.165877,-1.35,-1.40,-1.62,0.331754,0.689595,-0.689595,-33.0,405.0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,<NA>,0.0,0.0,<NA>,0,0
3,Doha,ATP250,Outdoor,Hard,1st Round,3.0,2015-01-05,Karlovic I.,Rosol L.,27.0,30.0,1320.0,1210.0,1.44,2.62,1.53,2.75,1.49,2.54,7.0,6.0,6.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1,2.0,0.076124,0.645320,0.354680,0.630273,0.369727,0.642523,0.357477,0.130273,-1.18,-1.05,-1.22,0.260546,0.533388,-0.533388,-3.0,110.0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,<NA>,0.0,0.0,<NA>,0,0
4,Doha,ATP250,Outdoor,Hard,1st Round,3.0,2015-01-05,Becker B.,Bolelli S.,40.0,52.0,973.0,810.0,1.61,2.20,1.76,2.25,1.67,2.15,3.0,6.0,6.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0,2.0,0.075663,0.577428,0.422572,0.562827,0.437173,0.561097,0.438903,0.062827,-0.59,-0.48,-0.49,0.125654,0.252644,-0.252644,-12.0,163.0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,0.0,<NA>,0.0,0.0,0,<NA>,0,NaN,NaN,<NA>,0.0,0.0,<NA>,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16056,Paris,Masters 1000,Indoor,Hard,Quarterfinals,3.0,2025-10-31,Shelton B.,Sinner J.,7.0,2.0,3820.0,10500.0,5.00,1.17,5.25,1.17,5.11,1.16,3.0,6.0,3.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0,2.0,0.054701,0.189627,0.810373,0.185008,0.814992,0.182243,0.817757,0.314992,3.83,3.95,4.08,-0.629984,-1.482779,1.482779,5.0,-6680.0,0.6,0.6,87.0,0.635036,4.0,11.0,45,1,2,1.0,0.9,240.0,0.789474,7.0,21.0,56,1,7,-0.4,-0.3,-0.154437,-3.0,-10.0,0,-11,7
16057,Paris,Masters 1000,Indoor,Hard,Quarterfinals,3.0,2025-10-31,Zverev A.,Medvedev D.,3.0,13.0,6160.0,2810.0,2.10,1.73,2.10,1.80,2.04,1.75,2.0,6.0,6.0,3.0,7.0,6.0,0.0,0.0,0.0,0.0,2.0,1.0,1,3.0,0.054225,0.451697,0.548303,0.461741,0.538259,0.461538,0.538462,0.038259,0.37,0.29,0.30,-0.076517,-0.153334,0.153334,-10.0,3350.0,0.8,0.7,397.0,0.714029,8.0,20.0,65,1,2,0.8,0.8,288.0,0.711111,5.0,22.0,49,1,3,0.0,-0.1,0.002918,3.0,-2.0,0,16,13
16